In [ ]:
# Test connection to a local MCP server
# Written for DSI Doctoral certificate, deploying AI module
# Robert Lu, July 12, 2026

from langgraph.graph import StateGraph, MessagesState, START
from langchain.chat_models import init_chat_model
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from langchain_core.messages import SystemMessage,  HumanMessage

import os
from openai import OpenAI

from dotenv import load_dotenv

load_dotenv("../.env")
load_dotenv("../.secrets")



True

In [2]:
if not os.environ.get("API_GATEWAY_KEY"):
    raise ValueError("Missing API_GATEWAY_KEY environment variable")
    
client = init_chat_model(
        "gpt-4o-mini",
        model_provider="openai",
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        api_key="any",
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client2 = get_client()

In [3]:
# example from https://developers.openai.com/api/docs/guides/tools-connectors-mcp?quickstart-panels=remote-mcp

resp = client2.responses.create(
    model="gpt-4o-mini",
    tools=[
        {
            "type": "mcp",
            "server_label": "dmcp",
            "server_description": "A Dungeons and Dragons MCP server to assist with dice rolling.",
            "server_url": "https://dmcp-server.deno.dev/sse",
            "require_approval": "never",
        },
    ],
    input="Roll 2d4+1",
)

In [5]:
print(resp.output_text)


You rolled a total of **6** for 2d4 + 1.


In [9]:
user_prompt = f"list the latest 5 papers added to my Zotero collection"

#client3 = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# needed pyzotero package, which unfortunately doesn't exist within the normal deployin_ai environment
# uv pip install pyzotero

# start up a local server, cd zotero-mcp-server/src
# run the command below
# npx mcp-proxy --port 8080 --tunnel -- python -m zotero_mcp.server
# need to set ZOTERO_LIBRARY_ID and ZOTERO_USER_ID in .zshrc

# get the local tunnel url, paste it below then append /sse to the end
local_tunnel_url = "https://big-sloth-49.tunnel.gla.ma/sse"
# then you can run this cell

#http://localhost:23119/api/
resp2 = client2.responses.create(
    model="gpt-4o-mini",
    tools=[
        {
            "type": "mcp",
            "server_label": "zotero-1",
            "server_description": "A local tool used to fetch, list, and search collections and items inside the user's Zotero desktop application reference library.",
            "server_url": local_tunnel_url,
            "require_approval": "never",
            #"headers": {
            #    "Authorization": f"Bearer {os.getenv('ZOTERO_MCP_TOKEN')}",
            #}
            
        },
    ],
    input=user_prompt,
)

In [10]:
print(resp2.output_text)

Here are the latest 5 papers added to your Zotero collection:

1. **Characterization techniques for heterogeneous nucleation from the gas phase**
   - **Authors:** Paul M. Winkler, Paul E. Wagner
   - **Date:** January 2022
   - **Abstract:** This paper reviews techniques for characterizing heterogeneous nucleation, which is essential for understanding aerosol particle formation and influences processes like cloud formation.
   - **DOI:** [10.1016/j.jaerosci.2021.105875](https://linkinghub.elsevier.com/retrieve/pii/S0021850221006042)

2. **A Review of Experimental Methods for Nucleation Rate Determination in Large-Volume Batch and Microfluidic Crystallization**
   - **Authors:** Cedric Devos, Tom Van Gerven, Simon Kuhn
   - **Date:** April 7, 2021
   - **Abstract:** The review discusses various methods for determining nucleation rates in crystallization, comparing traditional large-volume batch methods with newer microfluidic approaches.
   - **DOI:** [10.1021/acs.cgd.0c01606](https://

In [18]:
print(resp2.output_text)

Here are the latest 5 papers added to your Zotero collection:

1. **Characterization techniques for heterogeneous nucleation from the gas phase**
   - **Authors:** Paul M. Winkler, Paul E. Wagner
   - **Date:** January 2022
   - **Abstract:** Reviews various techniques for the characterization of heterogeneous nucleation, crucial in aerosol particle formation and nanoparticle detection.
   - **[Read here](https://linkinghub.elsevier.com/retrieve/pii/S0021850221006042)** | **DOI:** [10.1016/j.jaerosci.2021.105875](https://doi.org/10.1016/j.jaerosci.2021.105875)

2. **A Review of Experimental Methods for Nucleation Rate Determination in Large-Volume Batch and Microfluidic Crystallization**
   - **Authors:** Cedric Devos, Tom Van Gerven, Simon Kuhn
   - **Date:** April 7, 2021
   - **Abstract:** Discusses experimental nucleation rate determination methods, emphasizing large-volume batch crystallizers and microfluidic platforms.
   - **[Read here](https://pubs.acs.org/doi/10.1021/acs.cgd.0

In [14]:
print(resp2)

Response(id='resp_0609b9fa5595b43a006a543a7da92081a392f65a04d758e69d', created_at=1783904893.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[McpListTools(id='mcpl_0609b9fa5595b43a006a543a7dced081a3b34c623a0f03738a', server_label='zotero-2', tools=[McpListToolsTool(input_schema={'type': 'object', 'properties': {'query': {'description': 'Search query string', 'title': 'Query', 'type': 'string'}, 'limit': {'default': 10, 'description': 'Maximum number of results', 'maximum': 50, 'title': 'Limit', 'type': 'integer'}}, 'required': ['query'], 'title': 'SearchInput'}, name='search_papers', annotations={'read_only': False}, description='Search for papers in the Zotero library by keyword, title, or author'), McpListToolsTool(input_schema={'type': 'object', 'properties': {'arxiv_id': {'description': "arXiv paper ID (e.g., '2301.00234') or URL", 'title': 'Arxiv Id', 'type': 'string'}}, 'required': ['arxiv_id'], 'ti